# Fine-tunning

In [4]:
#modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
#embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
# from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from transformers import BertTokenizer, BertModel
#treino
from sklearn.model_selection import train_test_split
#metricas
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
#manipulacaoa
import pandas as pd
import numpy as np
#visualizacao
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.preprocessing import normalize


In [5]:
PATH_PARQUET = '/content/dataset_tidy.parquet'   # ajuste o caminho
PATH_ACCORD = '/content/accord_official_features.csv'   # ajuste o caminho
PATH_BASE = '/content/base_construida_features.csv'   # ajuste o caminho

In [6]:
df_parquet = pd.read_parquet(PATH_PARQUET)
df_accord = pd.read_csv(PATH_ACCORD)
df_base = pd.read_csv(PATH_BASE)
print('Bases de dados carregadas:')
print(f' - Parquet: {len(df_parquet)} amostras')
print(f' - ACCORD Official: {len(df_accord)} amostras')
print(f' - Base Construída: {len(df_base)} amostras')

Bases de dados carregadas:
 - Parquet: 2942 amostras
 - ACCORD Official: 2334 amostras
 - Base Construída: 2768 amostras


In [7]:
x_text_a = df_accord['Text']
y_a      = df_accord['is_rule'].values
X_train_text_a, X_test_text_a, y_train_a, y_test_a = train_test_split(x_text_a, y_a, test_size=0.2, random_state=42)

print(f'Treino : {len(y_train_a)} amostras '
      f'({y_train_a.sum()} regras / {(y_train_a==0).sum()} não-regras)')
print(f'Teste  : {len(y_test_a)} amostras '
      f'({y_test_a.sum()} regras / {(y_test_a==0).sum()} não-regras)')
print()
print(f'% regras no treino : {y_train_a.mean()*100:.1f}%')
print(f'% regras no teste  : {y_test_a.mean()*100:.1f}%')


Treino : 1867 amostras (606 regras / 1261 não-regras)
Teste  : 467 amostras (162 regras / 305 não-regras)

% regras no treino : 32.5%
% regras no teste  : 34.7%


In [8]:
x_text_b = df_base['Text']
y_b      = df_base['is_rule'].values
X_train_text_b, X_test_text_b, y_train_b, y_test_b = train_test_split(x_text_b, y_b, test_size=0.2, random_state=42)

print(f'Treino : {len(y_train_b)} amostras '
      f'({y_train_b.sum()} regras / {(y_train_b==0).sum()} não-regras)')
print(f'Teste  : {len(y_test_b)} amostras '
      f'({y_test_b.sum()} regras / {(y_test_b==0).sum()} não-regras)')
print()
print(f'% regras no treino : {y_train_b.mean()*100:.1f}%')
print(f'% regras no teste  : {y_test_b.mean()*100:.1f}%')


Treino : 2214 amostras (934.0 regras / 1280 não-regras)
Teste  : 554 amostras (246.0 regras / 308 não-regras)

% regras no treino : 42.2%
% regras no teste  : 44.4%


In [9]:
x_text_p = df_parquet['Text']
y_p      = df_parquet['is_rule'].values
X_train_text_p, X_test_text_p, y_train_p, y_test_p = train_test_split(x_text_p, y_p, test_size=0.2, random_state=42)

print(f'Treino : {len(y_train_p)} amostras '
      f'({y_train_p.sum()} regras / {(y_train_p==0).sum()} não-regras)')
print(f'Teste  : {len(y_test_p)} amostras '
      f'({y_test_p.sum()} regras / {(y_test_p==0).sum()} não-regras)')
print()
print(f'% regras no treino : {y_train_p.mean()*100:.1f}%')
print(f'% regras no teste  : {y_test_p.mean()*100:.1f}%')


Treino : 2353 amostras (985 regras / 1368 não-regras)
Teste  : 589 amostras (249 regras / 340 não-regras)

% regras no treino : 41.9%
% regras no teste  : 42.3%


In [10]:
device = "cuda"
MODEL_NAME = 'ACCORD-NLP/roberta-large-lm'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_roberta = AutoModel.from_pretrained(MODEL_NAME)
model_roberta = model_roberta.to(device)
model_roberta.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: ACCORD-NLP/roberta-large-lm
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 1024, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      

In [11]:
def get_embeddings_meanpool(texts, batch_size=32, max_length=128):
    all_embeddings = []
    n_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_num = i // batch_size + 1
        if batch_num % 5 == 0 or batch_num == 1:
            print(f'   Batch {batch_num}/{n_batches}')

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            output = model_roberta(**encoded)

        # Mean pooling — média dos tokens reais (ignora padding)
        attention_mask = encoded['attention_mask']
        token_embeddings = output.last_hidden_state  # (batch, seq_len, 1024)

        # Expande a mask para multiplicar com os embeddings
        mask_expanded = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        # Soma dos embeddings dos tokens reais
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        # Número de tokens reais por sentença
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        # Média
        mean_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
        all_embeddings.append(mean_embeddings)

    return np.vstack(all_embeddings)

In [12]:

emb_train_p_mp = get_embeddings_meanpool(X_train_text_p.tolist())
emb_test_p_mp  = get_embeddings_meanpool(X_test_text_p.tolist())

print(f'Variância mean pooling: {emb_train_p_mp.var(axis=0).mean():.6f}')
# Esperado: > 0.01, idealmente > 0.05'

   Batch 1/74


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

   Batch 5/74
   Batch 10/74
   Batch 15/74
   Batch 20/74
   Batch 25/74
   Batch 30/74
   Batch 35/74
   Batch 40/74
   Batch 45/74
   Batch 50/74
   Batch 55/74
   Batch 60/74
   Batch 65/74
   Batch 70/74
   Batch 1/19
   Batch 5/19
   Batch 10/19
   Batch 15/19
Variância mean pooling: 0.020831


In [13]:
emb_train_a_mp = get_embeddings_meanpool(X_train_text_a.tolist())
emb_test_a_mp  = get_embeddings_meanpool(X_test_text_a.tolist())


   Batch 1/59
   Batch 5/59
   Batch 10/59
   Batch 15/59
   Batch 20/59
   Batch 25/59
   Batch 30/59
   Batch 35/59
   Batch 40/59
   Batch 45/59
   Batch 50/59
   Batch 55/59
   Batch 1/15
   Batch 5/15
   Batch 10/15
   Batch 15/15


In [14]:
emb_train_b_mp = get_embeddings_meanpool(X_train_text_b.tolist())
emb_test_b_mp  = get_embeddings_meanpool(X_test_text_b.tolist())


   Batch 1/70
   Batch 5/70
   Batch 10/70
   Batch 15/70
   Batch 20/70
   Batch 25/70
   Batch 30/70
   Batch 35/70
   Batch 40/70
   Batch 45/70
   Batch 50/70
   Batch 55/70
   Batch 60/70
   Batch 65/70
   Batch 70/70
   Batch 1/18
   Batch 5/18
   Batch 10/18
   Batch 15/18


In [15]:
# ── Treino e avaliação — 3 modelos × 3 bases ─────────────────────────────────
from sklearn.preprocessing import StandardScaler

def treinar_com_embeddings(nome_base, emb_train, emb_test, y_train, y_test):
    # Normaliza os embeddings
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(emb_train)
    X_test_sc  = scaler.transform(emb_test)

    resultados = []

    # Regressão Logística
    lr = LogisticRegression(class_weight='balanced', max_iter=1000,
                             random_state=42, C=1.0)
    lr.fit(X_train_sc, y_train)
    y_pred = lr.predict(X_test_sc)
    print('═'*55)
    print(f'REGRESSÃO LOGÍSTICA — RoBERTa Embeddings ({nome_base})')
    print('═'*55)
    print(classification_report(y_test, y_pred, target_names=['Não-Regra','Regra']))
    resultados.append(('Logistic Regression', lr, X_test_sc, y_pred))

    # Random Forest
    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample',
                                 random_state=42, n_jobs=-1)
    rf.fit(X_train_sc, y_train)
    y_pred = rf.predict(X_test_sc)
    print('═'*55)
    print(f'RANDOM FOREST — RoBERTa Embeddings ({nome_base})')
    print('═'*55)
    print(classification_report(y_test, y_pred, target_names=['Não-Regra','Regra']))
    resultados.append(('Random Forest', rf, X_test_sc, y_pred))

    # SVM
    svm = SVC(kernel='linear', class_weight='balanced', probability=True,
               random_state=42, C=1.0)
    svm.fit(X_train_sc, y_train)
    y_pred = svm.predict(X_test_sc)
    print('═'*55)
    print(f'SVM — RoBERTa Embeddings ({nome_base})')
    print('═'*55)
    print(classification_report(y_test, y_pred, target_names=['Não-Regra','Regra']))
    resultados.append(('SVM', svm, X_test_sc, y_pred))

    return resultados, scaler

In [16]:
res_emb_p, scaler_p = treinar_com_embeddings('Parquet', emb_train_p_mp, emb_test_p_mp, y_train_p, y_test_p)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — RoBERTa Embeddings (Parquet)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.92      0.88      0.90       340
       Regra       0.84      0.90      0.87       249

    accuracy                           0.89       589
   macro avg       0.88      0.89      0.88       589
weighted avg       0.89      0.89      0.89       589

═══════════════════════════════════════════════════════
RANDOM FOREST — RoBERTa Embeddings (Parquet)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.86      0.92      0.89       340
       Regra       0.88      0.80      0.84       249

    accuracy                           0.87       589
   macro avg       0.87      0.86      0.87       589
weighted avg       0.87      0.87      0.87       589

══════════════════════════

In [17]:
res_emb_a, scaler_a = treinar_com_embeddings('ACCORD Official', emb_train_a_mp, emb_test_a_mp, y_train_a, y_test_a)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — RoBERTa Embeddings (ACCORD Official)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.91      0.91      0.91       305
       Regra       0.84      0.83      0.84       162

    accuracy                           0.89       467
   macro avg       0.88      0.87      0.87       467
weighted avg       0.89      0.89      0.89       467

═══════════════════════════════════════════════════════
RANDOM FOREST — RoBERTa Embeddings (ACCORD Official)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.79      0.98      0.87       305
       Regra       0.93      0.51      0.66       162

    accuracy                           0.82       467
   macro avg       0.86      0.74      0.77       467
weighted avg       0.84      0.82      0.80       467

══════════

In [18]:
res_emb_b, scaler_b = treinar_com_embeddings('Base Construída', emb_train_b_mp, emb_test_b_mp, y_train_b, y_test_b)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — RoBERTa Embeddings (Base Construída)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.91      0.89      0.90       308
       Regra       0.87      0.89      0.88       246

    accuracy                           0.89       554
   macro avg       0.89      0.89      0.89       554
weighted avg       0.89      0.89      0.89       554

═══════════════════════════════════════════════════════
RANDOM FOREST — RoBERTa Embeddings (Base Construída)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.83      0.89      0.86       308
       Regra       0.85      0.76      0.81       246

    accuracy                           0.84       554
   macro avg       0.84      0.83      0.83       554
weighted avg       0.84      0.84      0.83       554

══════════

In [19]:
# ── Summary table — RoBERTa Embeddings ───────────────────────────────────────
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
configs_emb = [
    (res_emb_p, 'Parquet',         y_test_p),
    (res_emb_a, 'ACCORD Official', y_test_a),
    (res_emb_b, 'Base Construída', y_test_b),
]

resultados_emb = []
for res, nome_base, y_test in configs_emb:
    for nome_modelo, model, X_test_sc, y_pred in res:
        resultados_emb.append({
            'Base':          nome_base,
            'Modelo':        nome_modelo,
            'Representação': 'RoBERTa-large (ACCORD)',
            'Accuracy':      round(accuracy_score(y_test, y_pred), 4),
            'Precision':     round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'Recall':        round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'F1-Regra':      round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'F1-NRegra':     round(f1_score(y_test, y_pred, pos_label=0, zero_division=0), 4),
            'F1-Macro':      round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
        })

df_summary_emb = pd.DataFrame(resultados_emb)

print('╔══ SUMMARY — RoBERTa ACCORD Embeddings ══════════════════════════════════╗')
print(df_summary_emb.to_string(index=False))
print('╚══════════════════════════════════════════════════════════════════════════╝')

print('\n🏆 Melhor por base:')
for base in df_summary_emb['Base'].unique():
    sub  = df_summary_emb[df_summary_emb['Base'] == base]
    best = sub.loc[sub['F1-Macro'].idxmax()]
    print(f'   {base:<20} → {best["Modelo"]:<22} F1-Macro={best["F1-Macro"]:.4f}')

df_summary_emb.to_csv('summary_roberta_embeddings.csv', index=False)

╔══ SUMMARY — RoBERTa ACCORD Embeddings ══════════════════════════════════╗
           Base              Modelo          Representação  Accuracy  Precision  Recall  F1-Regra  F1-NRegra  F1-Macro
        Parquet Logistic Regression RoBERTa-large (ACCORD)    0.8862     0.8421  0.8996    0.8699     0.8989    0.8844
        Parquet       Random Forest RoBERTa-large (ACCORD)    0.8710     0.8811  0.8032    0.8403     0.8917    0.8660
        Parquet                 SVM RoBERTa-large (ACCORD)    0.8625     0.8231  0.8594    0.8409     0.8789    0.8599
ACCORD Official Logistic Regression RoBERTa-large (ACCORD)    0.8865     0.8385  0.8333    0.8359     0.9133    0.8746
ACCORD Official       Random Forest RoBERTa-large (ACCORD)    0.8158     0.9318  0.5062    0.6560     0.8743    0.7651
ACCORD Official                 SVM RoBERTa-large (ACCORD)    0.8608     0.8129  0.7778    0.7950     0.8947    0.8448
Base Construída Logistic Regression RoBERTa-large (ACCORD)    0.8899     0.8685  0.8862    